In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

wings = ['A', 'B', 'C', 'D']
flat_ids = [
    f"{wing}-{floor}{num:02d}"
    for wing in wings
    for floor in range(1, 5)
    for num in range(1, 6)
]

flats_df = pd.DataFrame({
    'flat_id': flat_ids,
    'bhk': np.random.randint(1, 4, size=len(flat_ids)),
    'occupants': np.random.randint(1, 6, size=len(flat_ids))
})

bill_months = [
    '01/06/2026', '01/07/2026', '01/08/2026',
    '01/09/2026', '01/10/2026', '01/11/2026'
]

bills_base = pd.DataFrame([
    {'flat_id': row.flat_id, 'bill_month': month, 'bhk': row.bhk}
    for row in flats_df.itertuples(index=False)
    for month in bill_months
])

bills_raw = pd.DataFrame({
    'flat_id': bills_base['flat_id'],
    'bill_month': bills_base['bill_month'],
    'electricity_units': (
        bills_base['bhk'] * 100 + np.random.randint(20, 121, size=len(bills_base))
    ).astype(float),
    'water_kl': np.random.randint(8, 36, size=len(bills_base))
})

blank_idx = np.random.choice(bills_raw.index, size=15, replace=False)
bills_raw.loc[blank_idx, 'electricity_units'] = np.nan

bills_raw.iloc[[10, 110, 210, 310], bills_raw.columns.get_loc('electricity_units')] = 9999

bills_raw = pd.concat([bills_raw, bills_raw.iloc[:6]], ignore_index=True)

flats_df.to_csv('society_flats.csv', index=False)
bills_raw.to_csv('society_bills_raw.csv', index=False)

flats_df.head(), bills_raw.head()

(  flat_id  bhk  occupants
 0   A-101    3          4
 1   A-102    1          1
 2   A-103    3          5
 3   A-104    3          5
 4   A-105    1          2,
   flat_id  bill_month  electricity_units  water_kl
 0   A-101  01/06/2026              411.0        19
 1   A-101  01/07/2026              408.0        24
 2   A-101  01/08/2026              381.0        13
 3   A-101  01/09/2026              416.0        14
 4   A-101  01/10/2026              320.0        21)

In [4]:
bills_raw = pd.read_csv('society_bills_raw.csv')

bills_clean = bills_raw.drop_duplicates().copy()
bills_clean['electricity_units'] = bills_clean['electricity_units'].fillna(
    bills_clean['electricity_units'].median()
)
bills_clean['bill_month'] = pd.to_datetime(bills_clean['bill_month'], dayfirst=True)

q1 = bills_clean['electricity_units'].quantile(0.25)
q3 = bills_clean['electricity_units'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

bills_clean = bills_clean[
    (bills_clean['electricity_units'] >= lower_fence) &
    (bills_clean['electricity_units'] <= upper_fence)
].reset_index(drop=True)

print(bills_clean.shape)

(476, 4)


In [5]:
flats_df = pd.read_csv('society_flats.csv')

society_master = pd.merge(bills_clean, flats_df, on='flat_id', how='left')
society_master['wing'] = society_master['flat_id'].str.split('-').str[0]
society_master['month_no'] = society_master['bill_month'].dt.month
society_master['electricity_bill_inr'] = society_master['electricity_units'] * 8
society_master['units_per_person'] = (
    society_master['electricity_units'] / society_master['occupants']
).round(1)

assert len(society_master) == len(bills_clean)

society_master.to_csv('society_master.csv', index=False)
society_master.head()

,flat_id,bill_month,electricity_units,water_kl,bhk,occupants,wing,month_no,electricity_bill_inr,units_per_person
0,A-101,2026-06-01,411.0,19,3,4,A,6,3288.0,102.8
1,A-101,2026-07-01,408.0,24,3,4,A,7,3264.0,102.0
2,A-101,2026-08-01,381.0,13,3,4,A,8,3048.0,95.2
3,A-101,2026-09-01,416.0,14,3,4,A,9,3328.0,104.0
4,A-101,2026-10-01,320.0,21,3,4,A,10,2560.0,80.0


In [6]:
wing_month_pivot = society_master.pivot_table(
    values='electricity_units',
    index='wing',
    columns='month_no',
    aggfunc='mean'
).round(1)

top_wing = wing_month_pivot.mean(axis=1).idxmax()
top_month = int(wing_month_pivot.mean(axis=0).idxmax())

flat_upp_avg = society_master.groupby('flat_id')['units_per_person'].mean()
top_flat = flat_upp_avg.idxmax()

society_insights = pd.DataFrame({
    'question': [
        'Wing with highest average electricity units',
        'Month with highest average electricity units',
        'Flat with highest average units per person'
    ],
    'answer': [
        top_wing,
        top_month,
        top_flat
    ]
})

society_insights.to_csv('society_insights.csv', index=False)
wing_month_pivot, society_insights

(month_no     6      7      8      9      10     11
 wing                                              
 A         291.2  290.1  290.4  302.8  282.5  292.2
 B         278.2  286.1  282.2  275.3  287.2  275.0
 C         279.4  259.6  269.8  277.0  288.3  265.8
 D         280.8  289.2  275.6  277.1  271.8  268.0,
                                        question answer
 0   Wing with highest average electricity units      A
 1  Month with highest average electricity units      9
 2    Flat with highest average units per person  C-401)